# 8. Optional exploratory analyses

These legacy analyses are now available as notebook code cells rather than requiring a separate `.py` entry point. Each analysis is opt-in and reads the project data/results. Some write to fixed result filenames and can replace prior outputs; check each section's output path before enabling it. Some require earlier model/PCA artifacts; run only the relevant section after its inputs exist.

They are exploratory and development-oriented, not part of the active shared-profile model or independent test evaluation.


## 1. Common imports and project root


In [ ]:
from pathlib import Path
import json
import hashlib
import numpy as np
import pandas as pd

try:
    from IPython.display import Image, display
except ImportError:
    Image = None
    display = print

rootPath = Path.cwd()
while not (rootPath / 'data' / 'processed').exists() and rootPath != rootPath.parent:
    rootPath = rootPath.parent
if not (rootPath / 'data' / 'processed').exists(): raise FileNotFoundError('Open this notebook inside the project.')


## 2. Salary-value shortlist analysis

Set the switch to `True` to run this analysis. Prerequisite files and output location are defined in the implementation below.


In [ ]:
"""Compare PCA-based performance forecasts with cap-share salary on train/dev.

The salary-performance benchmark is fit on the training split only. Shortlist
evaluation is development-only; the previously examined test split is never
loaded. Rows with missing or source-review salaries are excluded from the
primary comparison and counted in the audit.
"""

from pathlib import Path
import hashlib
import json

import numpy as np
import pandas as pd

ROOT = rootPath
PROCESSED = ROOT / "data" / "processed"
RESULTS = ROOT / "results" / "development" / "salary_value"
PCA_PARAMETERS_PATH = ROOT / "models" / "pca" / "pca_parameters.json"
MODEL_DATA_PATH = PROCESSED / "model_data.npz"
SALARY_PATH = PROCESSED / "salary_cap_normalized_pairs.csv"
SHORTLIST_SIZES = (25, 50, 100)


def combine(values, parameters):
    values = np.asarray(values, dtype=np.float64)
    if values.ndim != 2 or values.shape[1] != len(parameters["target_order"]):
        raise ValueError("PCA target columns do not match the saved training-fitted PCA.")
    standardized = (
        values - np.asarray(parameters["target_means"])
    ) / np.asarray(parameters["target_stds"])
    centered = standardized - np.asarray(parameters["pca_center"])
    return (
        centered @ np.asarray(parameters["component_weights"])
        - parameters["score_mean"]
    ) / parameters["score_std"]


def checked_join(rows, salary, label):
    keys = ["player_id", "target_season"]
    salary_fields = keys + [
        "target_salary",
        "salary_cap_share",
        "salary_cap_share_pct",
        "salary_needs_source_review",
        "salary_method",
        "salary_available_check",
    ]
    joined = rows.merge(
        salary[salary_fields],
        on=keys,
        how="left",
        validate="one_to_one",
        indicator=True,
    )
    if not joined["_merge"].eq("both").all():
        raise ValueError(label + " rows did not all match normalized salary records.")
    return joined.drop(columns="_merge")


def fit_salary_benchmark(train):
    eligible = (
        train["salary_available_check"].fillna(False).astype(bool)
        & ~train["salary_needs_source_review"].fillna(True).astype(bool)
        & np.isfinite(train["actual_pca_score"])
        & np.isfinite(train["salary_cap_share"])
        & train["salary_cap_share"].gt(0)
    )
    fit_rows = train.loc[eligible].copy()
    if len(fit_rows) < 20:
        raise ValueError("Too few non-flagged training salaries to fit a benchmark.")

    # Log cap share captures a diminishing performance association while
    # keeping the benchmark transparent and trained entirely on training data.
    x = np.log(fit_rows["salary_cap_share"].to_numpy(dtype=float))
    y = fit_rows["actual_pca_score"].to_numpy(dtype=float)
    design = np.column_stack([np.ones(len(x)), x])
    coefficients, _, rank, _ = np.linalg.lstsq(design, y, rcond=None)
    if rank != 2 or not np.isfinite(coefficients).all():
        raise ValueError("Salary benchmark fit was singular or nonfinite.")
    return coefficients, fit_rows


def expected_performance(cap_share, coefficients):
    share = np.asarray(cap_share, dtype=float)
    result = np.full(share.shape, np.nan, dtype=float)
    valid = np.isfinite(share) & (share > 0)
    result[valid] = coefficients[0] + coefficients[1] * np.log(share[valid])
    return result


def rank_shortlists(dev, score_columns):
    rows = []
    for k in SHORTLIST_SIZES:
        available_k = min(k, len(dev))
        ranked = {}
        for label, column in score_columns.items():
            ranked[label] = set(
                dev.nlargest(available_k, column, keep="first").index.tolist()
            )
        actual_top = set(
            dev.nlargest(available_k, "actual_value_residual", keep="first").index.tolist()
        )
        for label, selected in ranked.items():
            chosen = dev.loc[list(selected)]
            rows.append(
                {
                    "shortlist_size": available_k,
                    "ranking": label,
                    "mean_actual_pca_score": float(chosen["actual_pca_score"].mean()),
                    "mean_actual_value_residual": float(chosen["actual_value_residual"].mean()),
                    "mean_salary_cap_share_pct": float(chosen["salary_cap_share_pct"].mean()),
                    "median_salary_cap_share_pct": float(chosen["salary_cap_share_pct"].median()),
                    "mean_salary_cap_share_pct_all_eligible": float(dev["salary_cap_share_pct"].mean()),
                    "overlap_with_actual_top_value": len(selected & actual_top),
                    "overlap_fraction_of_actual_top_value": len(selected & actual_top) / available_k,
                }
            )
    return pd.DataFrame(rows)


def runSalaryValueAnalysis():
    RESULTS.mkdir(parents=True, exist_ok=True)
    parameters = json.loads(PCA_PARAMETERS_PATH.read_text(encoding="utf-8"))
    salary = pd.read_csv(SALARY_PATH)
    data_hash = hashlib.sha256(MODEL_DATA_PATH.read_bytes()).hexdigest()
    if parameters["model_data_sha256"] != data_hash:
        raise ValueError("PCA parameters do not match current model arrays.")
    if parameters["fit_split"] != "train":
        raise ValueError("PCA parameters were not fitted on training data.")

    train_rows = pd.read_csv(PROCESSED / "train_rows.csv")
    dev_scores = pd.read_csv(ROOT / "results" / "development" / "pca_scores.csv")
    train = train_rows[["player", "player_id", "season", "target_season"]].copy()
    dev = dev_scores[
        [
            "player", "player_id", "season", "target_season",
            "actual_pca_score", "predicted_pca_score", "baseline_pca_score",
        ]
    ].copy()

    with np.load(MODEL_DATA_PATH, allow_pickle=False) as arrays:
        if len(train) != len(arrays["X_train"]):
            raise ValueError("Training metadata does not match training arrays.")
        train_targets = np.column_stack(
            [arrays["y_" + target + "_train"].astype(float) for target in parameters["target_order"]]
        )
    train["actual_pca_score"] = combine(train_targets, parameters)
    train = checked_join(train, salary, "Training")
    dev = checked_join(dev, salary, "Development")

    coefficients, fit_rows = fit_salary_benchmark(train)
    train["benchmark_expected_pca_score"] = expected_performance(
        train["salary_cap_share"], coefficients
    )
    dev["benchmark_expected_pca_score"] = expected_performance(
        dev["salary_cap_share"], coefficients
    )
    train["observed_value_residual"] = (
        train["actual_pca_score"] - train["benchmark_expected_pca_score"]
    )
    dev["actual_value_residual"] = (
        dev["actual_pca_score"] - dev["benchmark_expected_pca_score"]
    )
    dev["forecast_value_residual"] = (
        dev["predicted_pca_score"] - dev["benchmark_expected_pca_score"]
    )
    dev["prior_performance_value_residual"] = (
        dev["baseline_pca_score"] - dev["benchmark_expected_pca_score"]
    )

    train.to_csv(RESULTS / "training_salary_benchmark_rows.csv", index=False)
    dev.to_csv(RESULTS / "development_salary_value_rows_all.csv", index=False)

    eligible = (
        dev["salary_available_check"].fillna(False).astype(bool)
        & ~dev["salary_needs_source_review"].fillna(True).astype(bool)
        & np.isfinite(dev["actual_pca_score"])
        & np.isfinite(dev["predicted_pca_score"])
        & np.isfinite(dev["baseline_pca_score"])
        & np.isfinite(dev["salary_cap_share"])
        & dev["salary_cap_share"].gt(0)
    )
    primary = dev.loc[eligible].copy()
    if len(primary) < max(SHORTLIST_SIZES):
        raise ValueError("Not enough non-flagged development salary rows for all shortlist sizes.")
    primary.to_csv(RESULTS / "development_salary_value_rows.csv", index=False)
    comparisons = rank_shortlists(
        primary,
        {
            "PCA forecast value": "forecast_value_residual",
            "prior-season PCA value": "prior_performance_value_residual",
        },
    )
    comparisons.to_csv(RESULTS / "shortlist_comparison.csv", index=False)

    model_rmse = float(np.sqrt(np.mean(
        (primary["predicted_pca_score"] - primary["actual_pca_score"]) ** 2
    )))
    baseline_rmse = float(np.sqrt(np.mean(
        (primary["baseline_pca_score"] - primary["actual_pca_score"]) ** 2
    )))
    audit = {
        "split_policy": "Benchmark fitted on training only; shortlist comparisons evaluated on development only; test split not loaded.",
        "performance_score": "Existing training-fitted PCA of next-season PIE, WS, and VORP; higher is better.",
        "salary_benchmark": "OLS linear fit of actual training PCA score on log(target-season salary / target-season salary cap), using available salaries not flagged for source review.",
        "benchmark_coefficients": {
            "intercept": float(coefficients[0]),
            "log_cap_share": float(coefficients[1]),
        },
        "benchmark_training_rows": int(len(fit_rows)),
        "training_rows_total": int(len(train)),
        "training_salary_available_not_flagged": int(
            (train["salary_available_check"].fillna(False).astype(bool)
             & ~train["salary_needs_source_review"].fillna(True).astype(bool)).sum()
        ),
        "development_rows_total": int(len(dev)),
        "development_salary_available_not_flagged": int(eligible.sum()),
        "development_salary_missing": int((~dev["salary_available_check"].fillna(False).astype(bool)).sum()),
        "development_salary_available_flagged_for_source_review": int(
            (dev["salary_available_check"].fillna(False).astype(bool)
             & dev["salary_needs_source_review"].fillna(True).astype(bool)).sum()
        ),
        "development_pca_rmse_on_nonflagged_salary_rows": model_rmse,
        "development_prior_pca_rmse_on_nonflagged_salary_rows": baseline_rmse,
        "development_rows_excluded_for_missing_or_review_salary": int((~eligible).sum()),
        "shortlist_sizes": list(SHORTLIST_SIZES),
        "comparison_csv": str((RESULTS / "shortlist_comparison.csv").relative_to(ROOT)),
        "interpretation": "Value residual is performance above/below the training-only average associated with cap-share salary; it is not causal or a contract valuation. Shortlist findings are exploratory because development data informed prior model selection.",
        "test_evaluated": False,
        "salary_normalized_file_sha256": hashlib.sha256(SALARY_PATH.read_bytes()).hexdigest(),
        "pca_parameters_sha256": hashlib.sha256(PCA_PARAMETERS_PATH.read_bytes()).hexdigest(),
    }
    (RESULTS / "analysis_audit.json").write_text(
        json.dumps(audit, indent=2, allow_nan=False) + "\n", encoding="utf-8"
    )

    print("Training-only salary benchmark rows: " + str(len(fit_rows)))
    print("Eligible (salary available, not flagged) development rows: " + str(len(primary)) + " / " + str(len(dev)))
    print("Development PCA RMSE / prior baseline RMSE: "
          + str(round(model_rmse, 4)) + " / " + str(round(baseline_rmse, 4)))
    print(comparisons.to_string(index=False))
    print("Test split not loaded. Results: " + str(RESULTS))

doSalaryValueAnalysis = False
if doSalaryValueAnalysis: runSalaryValueAnalysis()
print('Salary-value shortlist analysis enabled: ' + str(doSalaryValueAnalysis))

## 2. Historical player comps

Set the switch to `True` to run this analysis. Prerequisite files and output location are defined in the implementation below.


In [ ]:
"""Find historical player-season analogs and summarize their next-season jumps.

PCA and salary scaling are fit on training inputs only. Development queries use
the nearest distinct players whose observed target seasons were already complete
by the query player's prior season. The test split is never loaded.
"""

from pathlib import Path
import hashlib
import json

import numpy as np
import pandas as pd
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

ROOT = rootPath
PROCESSED = ROOT / "data" / "processed"
RESULTS = ROOT / "results" / "development" / "player_comps"
MODEL_DATA = PROCESSED / "model_data.npz"
SALARY_DATA = PROCESSED / "salary_cap_normalized_pairs.csv"
N_COMPONENTS = 10
PCA_VARIANCE = 0.90
SEED = 42
TARGETS = ["pie", "ws", "vorp"]


def attach_prior_salary(rows, salary):
    prior_salary = salary[
        [
            "player_id", "target_season", "salary_cap_share",
            "salary_available_check", "salary_needs_source_review",
        ]
    ].rename(
        columns={
            "target_season": "season",
            "salary_cap_share": "prior_salary_cap_share",
            "salary_available_check": "prior_salary_available",
            "salary_needs_source_review": "prior_salary_flagged",
        }
    )
    joined = rows.merge(
        prior_salary,
        on=["player_id", "season"],
        how="left",
        validate="one_to_one",
    )
    joined["prior_salary_cap_share"] = pd.to_numeric(
        joined["prior_salary_cap_share"], errors="coerce"
    )
    joined["prior_salary_available"] = (
        joined["prior_salary_available"].fillna(False).astype(bool)
    )
    joined["prior_salary_flagged"] = joined["prior_salary_flagged"].fillna(True).astype(bool)
    return joined


def runPlayerComps():
    RESULTS.mkdir(parents=True, exist_ok=True)
    salary = pd.read_csv(SALARY_DATA)
    train_rows = pd.read_csv(PROCESSED / "train_rows.csv")
    dev_rows = pd.read_csv(PROCESSED / "dev_rows.csv")
    with np.load(MODEL_DATA, allow_pickle=False) as arrays:
        train_x = arrays["X_train"].astype(float)
        dev_x = arrays["X_dev"].astype(float)
        if len(train_rows) != len(train_x) or len(dev_rows) != len(dev_x):
            raise ValueError("Metadata rows do not align with saved train/development inputs.")
        train = train_rows[
            ["player", "player_id", "season", "target_season", *TARGETS]
        ].copy()
        dev = dev_rows[
            ["player", "player_id", "season", "target_season", *TARGETS,
             "next_season_pie", "next_season_ws", "next_season_vorp"]
        ].copy()
        train_outcomes = np.column_stack(
            [arrays["y_" + target + "_train"].astype(float) for target in TARGETS]
        )

    train = attach_prior_salary(train, salary)
    dev = attach_prior_salary(dev, salary)
    train["prior_salary_cap_share"] = train["prior_salary_cap_share"].where(
        train["prior_salary_cap_share"].gt(0)
    )
    dev["prior_salary_cap_share"] = dev["prior_salary_cap_share"].where(
        dev["prior_salary_cap_share"].gt(0)
    )
    train["log_prior_salary_cap_share"] = np.log(train["prior_salary_cap_share"])
    dev["log_prior_salary_cap_share"] = np.log(dev["prior_salary_cap_share"])

    train_valid = (
        train["log_prior_salary_cap_share"].notna()
        & np.isfinite(train_x).all(axis=1)
        & np.isfinite(train_outcomes).all(axis=1)
    )
    dev_valid = (
        dev["log_prior_salary_cap_share"].notna()
        & np.isfinite(dev_x).all(axis=1)
        & np.isfinite(dev[["next_season_" + target for target in TARGETS]].to_numpy(float)).all(axis=1)
    )
    train_fit = train.loc[train_valid].reset_index(drop=True)
    dev_fit = dev.loc[dev_valid].reset_index(drop=True)
    train_x = train_x[train_valid]
    dev_x = dev_x[dev_valid]
    train_outcomes = train_outcomes[train_valid]
    if len(train_fit) <= N_COMPONENTS:
        raise ValueError("Too few training rows with prior salary for PCA.")

    salary_scaler = StandardScaler()
    train_salary = salary_scaler.fit_transform(
        train_fit[["log_prior_salary_cap_share"]].to_numpy(float)
    )
    dev_salary = salary_scaler.transform(
        dev_fit[["log_prior_salary_cap_share"]].to_numpy(float)
    )
    train_features = np.column_stack([train_x, train_salary])
    dev_features = np.column_stack([dev_x, dev_salary])
    pca = PCA(n_components=PCA_VARIANCE, svd_solver="full", random_state=SEED)
    train_pc = pca.fit_transform(train_features)
    dev_pc = pca.transform(dev_features)

    deltas = train_outcomes - train_fit[TARGETS].to_numpy(float)
    train_fit[[f"next_season_delta_{target}" for target in TARGETS]] = deltas

    neighbor_records = []
    prediction_records = []
    for query_index, query in dev_fit.iterrows():
        # Historical analog outcomes must have been observed by the query's
        # prior season; the query player is not allowed to match themself.
        eligible = (
            train_fit["target_season"].le(query["season"])
            & train_fit["player_id"].ne(query["player_id"])
        ).to_numpy()
        candidate_indices = np.flatnonzero(eligible)
        if not len(candidate_indices):
            continue
        distances = np.linalg.norm(train_pc[candidate_indices] - dev_pc[query_index], axis=1)
        ranked = candidate_indices[np.argsort(distances, kind="stable")]
        # Avoid several seasons from one analog player dominating the forecast.
        chosen = []
        seen_players = set()
        for idx in ranked:
            player_id = train_fit.iloc[idx]["player_id"]
            if player_id in seen_players:
                continue
            chosen.append(idx)
            seen_players.add(player_id)
            if len(chosen) == N_COMPONENTS:
                break
        if not chosen:
            continue

        chosen_distances = np.linalg.norm(train_pc[chosen] - dev_pc[query_index], axis=1)
        chosen_deltas = deltas[chosen]
        median_delta = np.median(chosen_deltas, axis=0)
        prior = query[TARGETS].to_numpy(float)
        forecast = prior + median_delta
        pred = {
            "player": query["player"],
            "player_id": query["player_id"],
            "season": int(query["season"]),
            "target_season": int(query["target_season"]),
            "analogs_used": int(len(chosen)),
            "median_analog_distance": float(np.median(chosen_distances)),
            "prior_salary_cap_share": float(query["prior_salary_cap_share"]),
            "prior_salary_flagged": bool(query["prior_salary_flagged"]),
        }
        for j, target in enumerate(TARGETS):
            pred[f"prior_{target}"] = float(prior[j])
            pred[f"predicted_delta_{target}"] = float(median_delta[j])
            pred[f"predicted_next_{target}"] = float(forecast[j])
            pred[f"actual_next_{target}"] = float(query["next_season_" + target])
            pred[f"actual_delta_{target}"] = float(query["next_season_" + target] - prior[j])
        prediction_records.append(pred)

        for rank, (idx, distance) in enumerate(zip(chosen, chosen_distances), start=1):
            analog = train_fit.iloc[idx]
            record = {
                "query_player": query["player"],
                "query_player_id": query["player_id"],
                "query_season": int(query["season"]),
                "query_target_season": int(query["target_season"]),
                "analog_rank": rank,
                "distance_in_retained_pca_space": float(distance),
                "analog_player": analog["player"],
                "analog_player_id": analog["player_id"],
                "analog_prior_season": int(analog["season"]),
                "analog_outcome_season": int(analog["target_season"]),
                "analog_prior_salary_cap_share": float(analog["prior_salary_cap_share"]),
                "analog_salary_flagged": bool(analog["prior_salary_flagged"]),
            }
            for j, target in enumerate(TARGETS):
                record[f"analog_prior_{target}"] = float(analog[target])
                record[f"analog_next_{target}"] = float(train_outcomes[idx, j])
                record[f"analog_delta_{target}"] = float(deltas[idx, j])
            neighbor_records.append(record)

    predictions = pd.DataFrame(prediction_records)
    neighbors = pd.DataFrame(neighbor_records)
    predictions.to_csv(RESULTS / "development_comp_predictions.csv", index=False)
    neighbors.to_csv(RESULTS / "development_comp_neighbors.csv", index=False)

    metric_rows = []
    for target in TARGETS:
        actual = predictions[f"actual_next_{target}"].to_numpy(float)
        analog = predictions[f"predicted_next_{target}"].to_numpy(float)
        persistence = predictions[f"prior_{target}"].to_numpy(float)
        metric_rows.append(
            {
                "target": target,
                "development_rows": int(len(predictions)),
                "analog_median_delta_rmse": float(np.sqrt(np.mean((analog - actual) ** 2))),
                "analog_median_delta_mae": float(np.mean(np.abs(analog - actual))),
                "persistence_rmse": float(np.sqrt(np.mean((persistence - actual) ** 2))),
                "persistence_mae": float(np.mean(np.abs(persistence - actual))),
            }
        )
    metrics = pd.DataFrame(metric_rows)
    metrics.to_csv(RESULTS / "development_comp_metrics.csv", index=False)
    metadata = {
        "method": "PCA-reduced Euclidean nearest neighbors; median observed next-season change among up to 10 distinct-player analogs",
        "pca_variance_retained": PCA_VARIANCE,
        "pca_components_retained": int(pca.n_components_),
        "explained_variance_ratio": pca.explained_variance_ratio_.tolist(),
        "feature_inputs": "Existing 22 standardized prior-season player features plus standardized log prior-season salary-cap share",
        "salary_timing": "Prior-season salary only, joined where salary target_season equals the model row's prior season; flagged available values included, missing values excluded",
        "analog_time_rule": "Analog target season must be no later than the query player's prior season",
        "distinct_analog_players": True,
        "max_analogs_per_query": N_COMPONENTS,
        "training_rows_total": int(len(train)),
        "training_rows_used": int(len(train_fit)),
        "development_rows_total": int(len(dev)),
        "development_rows_with_prior_salary": int(len(dev_fit)),
        "development_rows_with_at_least_one_historical_analog": int(len(predictions)),
        "development_rows_without_historical_analog": int(len(dev_fit) - len(predictions)),
        "training_salary_flagged_rows_included": int(
            train_fit["prior_salary_flagged"].sum()
        ),
        "development_salary_flagged_rows_used": int(
            predictions["prior_salary_flagged"].sum()
        ) if len(predictions) else 0,
        "test_data_loaded": False,
        "model_arrays_sha256": hashlib.sha256(MODEL_DATA.read_bytes()).hexdigest(),
        "salary_data_sha256": hashlib.sha256(SALARY_DATA.read_bytes()).hexdigest(),
        "interpretation": "Exploratory player-comparison forecasts; development results are not independent because development informed earlier project choices.",
    }
    (RESULTS / "analysis_audit.json").write_text(
        json.dumps(metadata, indent=2, allow_nan=False) + "\n", encoding="utf-8"
    )
    print(f"Retained PCA components: {pca.n_components_} ({pca.explained_variance_ratio_.sum():.1%} variance)")
    print(f"Training analog pool: {len(train_fit)} rows")
    print(f"Development queries with historical analogs: {len(predictions)} / {len(dev)}")
    print(f"Outputs: {RESULTS}")
    print(metrics.to_string(index=False))

doPlayerComps = False
if doPlayerComps: runPlayerComps()
print('Historical player comps enabled: ' + str(doPlayerComps))

## 3. PCA profile clusters

Set the switch to `True` to run this analysis. Prerequisite files and output location are defined in the implementation below.


In [ ]:
"""Cluster PCA-based performance profiles using training-only fit.

The scaler, PCA, and K-means clusters are fitted on actual training outcomes.
Development player profiles are assigned from the existing model forecasts;
observed development outcomes are used only for descriptive validation.
Salary is joined only for summaries and is not a PCA or clustering feature.
No test data are loaded.
"""

from pathlib import Path
import hashlib
import json

import numpy as np
import pandas as pd
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import adjusted_rand_score, silhouette_score
from sklearn.preprocessing import StandardScaler

ROOT = rootPath
PROCESSED = ROOT / "data" / "processed"
RESULTS = ROOT / "results" / "development" / "pca_profiles"
MODEL_DATA = PROCESSED / "model_data.npz"
PCA_PARAMETERS = ROOT / "models" / "pca" / "pca_parameters.json"
DEV_SCORES = ROOT / "results" / "development" / "pca_scores.csv"
SALARY_DATA = PROCESSED / "salary_cap_normalized_pairs.csv"
TARGETS = ["pie", "ws", "vorp"]
N_PROFILES = 4
SEED = 42


def runPcaProfileClusters():
    RESULTS.mkdir(parents=True, exist_ok=True)
    saved_parameters = json.loads(PCA_PARAMETERS.read_text(encoding="utf-8"))
    model_hash = hashlib.sha256(MODEL_DATA.read_bytes()).hexdigest()
    if saved_parameters["model_data_sha256"] != model_hash:
        raise ValueError("Saved PCA and current model arrays do not match.")

    with np.load(MODEL_DATA, allow_pickle=False) as arrays:
        training = np.column_stack(
            [arrays["y_" + target + "_train"].astype(float) for target in TARGETS]
        )
        if not np.isfinite(training).all():
            raise ValueError("Training outcome matrix contains missing/nonfinite values.")

    scaler = StandardScaler()
    train_standardized = scaler.fit_transform(training)
    pca = PCA(n_components=len(TARGETS), svd_solver="full")
    train_coordinates = pca.fit_transform(train_standardized)
    # PCA axis signs are arbitrary; orient PC1 toward higher average outcomes.
    if pca.components_[0].sum() < 0:
        pca.components_[0] *= -1
        train_coordinates[:, 0] *= -1

    clusterer = KMeans(n_clusters=N_PROFILES, random_state=SEED, n_init=50)
    train_cluster_raw = clusterer.fit_predict(train_coordinates)
    centers = clusterer.cluster_centers_
    order = np.argsort(centers[:, 0])
    profile_names = {int(cluster): f"Profile {rank + 1}" for rank, cluster in enumerate(order)}
    train_profiles = np.array([profile_names[int(value)] for value in train_cluster_raw])

    dev = pd.read_csv(DEV_SCORES)
    predicted_targets = dev[["predicted_" + target for target in TARGETS]].to_numpy(float)
    actual_targets = pd.read_csv(PROCESSED / "dev_rows.csv")[
        ["next_season_" + target for target in TARGETS]
    ].to_numpy(float)
    if len(dev) != len(actual_targets) or not np.isfinite(predicted_targets).all():
        raise ValueError("Development predictions and outcomes are missing or misaligned.")
    if not np.isfinite(actual_targets).all():
        raise ValueError("Development actual outcomes contain missing/nonfinite values.")

    predicted_coordinates = pca.transform(scaler.transform(predicted_targets))
    actual_coordinates = pca.transform(scaler.transform(actual_targets))
    predicted_cluster_raw = clusterer.predict(predicted_coordinates)
    actual_cluster_raw = clusterer.predict(actual_coordinates)
    predicted_profiles = np.array([profile_names[int(value)] for value in predicted_cluster_raw])
    actual_profiles = np.array([profile_names[int(value)] for value in actual_cluster_raw])

    players = dev[["player", "player_id", "season", "target_season"]].copy()
    for index, target in enumerate(TARGETS):
        players["predicted_" + target] = predicted_targets[:, index]
        players["actual_" + target] = actual_targets[:, index]
    for index in range(len(TARGETS)):
        players[f"predicted_pc{index + 1}"] = predicted_coordinates[:, index]
        players[f"actual_pc{index + 1}"] = actual_coordinates[:, index]
    players["predicted_profile"] = predicted_profiles
    players["actual_outcome_profile"] = actual_profiles
    players["profile_match"] = predicted_profiles == actual_profiles

    salary = pd.read_csv(SALARY_DATA)
    salary_fields = [
        "player_id", "target_season", "salary_cap_share_pct",
        "salary_available_check", "salary_needs_source_review",
    ]
    players = players.merge(
        salary[salary_fields],
        on=["player_id", "target_season"],
        how="left",
        validate="one_to_one",
    )
    players.to_csv(RESULTS / "development_player_profiles.csv", index=False)

    profile_rows = []
    for profile in sorted(profile_names.values(), key=lambda name: int(name.split()[-1])):
        selected = players[players["predicted_profile"] == profile]
        outcome_match = selected["profile_match"].mean()
        row = {
            "predicted_profile": profile,
            "players_assigned_by_forecast": int(len(selected)),
            "players_with_observed_outcome_in_profile": int(
                (players["actual_outcome_profile"] == profile).sum()
            ),
            "mean_predicted_pc1": float(selected["predicted_pc1"].mean()),
            "mean_actual_pc1": float(selected["actual_pc1"].mean()),
            "profile_assignment_match_rate": float(outcome_match),
            "mean_salary_cap_share_pct_available": float(
                selected["salary_cap_share_pct"].mean()
            ),
            "salary_rows_flagged_for_source_review": int(
                (selected["salary_needs_source_review"].fillna(True).astype(bool)
                 & selected["salary_available_check"].fillna(False).astype(bool)).sum()
            ),
        }
        for target in TARGETS:
            row["mean_predicted_" + target] = float(selected["predicted_" + target].mean())
            row["mean_actual_" + target] = float(selected["actual_" + target].mean())
        profile_rows.append(row)
    pd.DataFrame(profile_rows).to_csv(RESULTS / "profile_summary.csv", index=False)

    transition = pd.crosstab(
        pd.Series(actual_profiles, name="actual_outcome_profile"),
        pd.Series(predicted_profiles, name="predicted_profile"),
        normalize="index",
    )
    transition.to_csv(RESULTS / "profile_transition_rates.csv")

    model_metadata = {
        "targets": TARGETS,
        "training_rows": int(len(training)),
        "training_means": scaler.mean_.tolist(),
        "training_scales": scaler.scale_.tolist(),
        "pca_components": pca.components_.tolist(),
        "pca_explained_variance_ratio": pca.explained_variance_ratio_.tolist(),
        "cluster_count": N_PROFILES,
        "random_seed": SEED,
        "training_cluster_centers": centers.tolist(),
        "profile_order": "Profiles are numbered by ascending training cluster-center PC1; clusters are multidimensional performance profiles, not equal-width tiers.",
        "salary_used_as_pca_or_clustering_feature": False,
        "test_data_loaded": False,
    }
    (RESULTS / "fitted_profile_model.json").write_text(
        json.dumps(model_metadata, indent=2, allow_nan=False) + "\n",
        encoding="utf-8",
    )
    audit = {
        "split_policy": "Scaler, PCA, and K-means fitted on actual training outcomes only; development assignments use existing PCA model forecasts; test split not loaded.",
        "training_rows": int(len(training)),
        "development_rows": int(len(players)),
        "development_profile_match_rate": float(players["profile_match"].mean()),
        "development_adjusted_rand_index": float(
            adjusted_rand_score(actual_profiles, predicted_profiles)
        ),
        "training_silhouette_score": float(
            silhouette_score(train_coordinates, train_cluster_raw)
        ),
        "salary_is_context_only": True,
        "n_profiles": N_PROFILES,
        "test_evaluated": False,
        "model_data_sha256": model_hash,
    }
    (RESULTS / "analysis_audit.json").write_text(
        json.dumps(audit, indent=2, allow_nan=False) + "\n", encoding="utf-8"
    )
    print("Training-fitted PCA/K-means profiles: " + str(N_PROFILES))
    print("Training silhouette score: " + str(round(audit["training_silhouette_score"], 4)))
    print("Development forecast profile match rate vs observed outcomes: "
          + str(round(audit["development_profile_match_rate"], 4)))
    print("Development adjusted Rand index: "
          + str(round(audit["development_adjusted_rand_index"], 4)))
    print(pd.DataFrame(profile_rows).to_string(index=False))
    print("Salary excluded from clustering; test split not loaded.")
    print("Results: " + str(RESULTS))

doPcaProfileClusters = False
if doPcaProfileClusters: runPcaProfileClusters()
print('PCA profile clusters enabled: ' + str(doPcaProfileClusters))

## 4. Salary-aware PCA clusters

Set the switch to `True` to run this analysis. Prerequisite files and output location are defined in the implementation below.


In [ ]:
"""Find salary-aware player-season groups with a training-only PCA/K-means fit.

The four PCA inputs are next-season PIE, WS, VORP, and log salary-cap share.
The scaler, PCA, and K-means model use training rows only. Development rows
are projected and assigned for descriptive follow-up; the test split is not
loaded. Rows without a listed salary are omitted rather than imputed.
"""

from pathlib import Path
import hashlib
import json

import numpy as np
import pandas as pd
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score
from sklearn.preprocessing import StandardScaler

ROOT = rootPath
PROCESSED = ROOT / "data" / "processed"
RESULTS = ROOT / "results" / "development" / "salary_aware_pca"
MODEL_DATA = PROCESSED / "model_data.npz"
SALARY_DATA = PROCESSED / "salary_cap_normalized_pairs.csv"
N_PROFILES = 4
SEED = 42
PERFORMANCE = ["pie", "ws", "vorp"]
FEATURES = PERFORMANCE + ["log_salary_cap_share"]


def add_salary(rows, salary):
    fields = [
        "player_id", "target_season", "salary_cap_share",
        "salary_available_check", "salary_needs_source_review",
    ]
    joined = rows.merge(
        salary[fields],
        on=["player_id", "target_season"],
        how="left",
        validate="one_to_one",
    )
    joined["salary_cap_share"] = pd.to_numeric(joined["salary_cap_share"], errors="coerce")
    joined["salary_available_check"] = joined["salary_available_check"].fillna(False).astype(bool)
    joined["salary_needs_source_review"] = joined["salary_needs_source_review"].fillna(True).astype(bool)
    joined["log_salary_cap_share"] = np.nan
    positive = joined["salary_cap_share"].gt(0) & np.isfinite(joined["salary_cap_share"])
    joined.loc[positive, "log_salary_cap_share"] = np.log(joined.loc[positive, "salary_cap_share"])
    return joined


def runSalaryAwareClusters():
    RESULTS.mkdir(parents=True, exist_ok=True)
    salary = pd.read_csv(SALARY_DATA)
    train_rows = pd.read_csv(PROCESSED / "train_rows.csv")
    dev_rows = pd.read_csv(PROCESSED / "dev_rows.csv")

    with np.load(MODEL_DATA, allow_pickle=False) as arrays:
        if len(train_rows) != len(arrays["y_pie_train"]):
            raise ValueError("Training rows do not align with the saved training targets.")
        train = train_rows[["player", "player_id", "season", "target_season"]].copy()
        for target in PERFORMANCE:
            train[target] = arrays["y_" + target + "_train"].astype(float)
    dev = dev_rows[["player", "player_id", "season", "target_season"]].copy()
    for target in PERFORMANCE:
        dev[target] = pd.to_numeric(dev_rows["next_season_" + target], errors="coerce")

    train = add_salary(train, salary)
    dev = add_salary(dev, salary)
    train_eligible = (
        train[FEATURES].notna().all(axis=1)
        & np.isfinite(train[FEATURES].to_numpy(float)).all(axis=1)
    )
    dev_eligible = (
        dev[FEATURES].notna().all(axis=1)
        & np.isfinite(dev[FEATURES].to_numpy(float)).all(axis=1)
    )
    train_fit = train.loc[train_eligible].copy()
    dev_fit = dev.loc[dev_eligible].copy()
    if len(train_fit) <= N_PROFILES:
        raise ValueError("Too few eligible training rows for salary-aware profiles.")

    scaler = StandardScaler()
    train_scaled = scaler.fit_transform(train_fit[FEATURES].to_numpy(float))
    pca = PCA(n_components=len(FEATURES), svd_solver="full")
    train_pc = pca.fit_transform(train_scaled)
    if pca.components_[0].sum() < 0:
        pca.components_[0] *= -1
        train_pc[:, 0] *= -1

    clusterer = KMeans(n_clusters=N_PROFILES, random_state=SEED, n_init=50)
    train_labels = clusterer.fit_predict(train_pc)
    centers = clusterer.cluster_centers_
    order = np.argsort(centers[:, 0])
    profile_names = {int(label): f"Profile {rank + 1}" for rank, label in enumerate(order)}
    train_fit["profile"] = [profile_names[int(label)] for label in train_labels]
    train_fit[[f"pc{i + 1}" for i in range(len(FEATURES))]] = train_pc

    dev_scaled = scaler.transform(dev_fit[FEATURES].to_numpy(float))
    dev_pc = pca.transform(dev_scaled)
    dev_labels = clusterer.predict(dev_pc)
    dev_fit["profile"] = [profile_names[int(label)] for label in dev_labels]
    dev_fit[[f"pc{i + 1}" for i in range(len(FEATURES))]] = dev_pc
    train_fit["split"] = "training"
    dev_fit["split"] = "development"

    rows = pd.concat([train_fit, dev_fit], ignore_index=True)
    rows.to_csv(RESULTS / "salary_aware_player_groups.csv", index=False)

    summaries = []
    for profile in sorted(profile_names.values(), key=lambda name: int(name.split()[-1])):
        for split, group in rows.groupby("split"):
            selected = group[group["profile"].eq(profile)]
            if selected.empty:
                continue
            summary = {
                "split": split,
                "profile": profile,
                "player_seasons": int(len(selected)),
                "mean_salary_cap_share_pct": float(selected["salary_cap_share"].mean() * 100),
                "median_salary_cap_share_pct": float(selected["salary_cap_share"].median() * 100),
                "flagged_salary_rows": int(
                    (selected["salary_available_check"] & selected["salary_needs_source_review"]).sum()
                ),
                "pc1_center": float(selected["pc1"].mean()),
            }
            for target in PERFORMANCE:
                summary[f"mean_{target}"] = float(selected[target].mean())
            summaries.append(summary)
    pd.DataFrame(summaries).to_csv(RESULTS / "profile_summary.csv", index=False)

    metadata = {
        "features": FEATURES,
        "salary_feature_definition": "natural log of target-season salary divided by target-season salary cap",
        "salary_review_flags_included": "yes; current listed values retained, counts exported; missing salaries excluded, no imputation",
        "training_rows_total": int(len(train)),
        "training_rows_used": int(len(train_fit)),
        "training_missing_salary_or_outcome": int(len(train) - len(train_fit)),
        "training_salary_flagged_rows_used": int(
            (train_fit["salary_available_check"] & train_fit["salary_needs_source_review"]).sum()
        ),
        "development_rows_total": int(len(dev)),
        "development_rows_used": int(len(dev_fit)),
        "development_missing_salary_or_outcome": int(len(dev) - len(dev_fit)),
        "development_salary_flagged_rows_used": int(
            (dev_fit["salary_available_check"] & dev_fit["salary_needs_source_review"]).sum()
        ),
        "n_profiles": N_PROFILES,
        "seed": SEED,
        "scaler_means": scaler.mean_.tolist(),
        "scaler_scales": scaler.scale_.tolist(),
        "pca_components": pca.components_.tolist(),
        "explained_variance_ratio": pca.explained_variance_ratio_.tolist(),
        "cluster_centers": centers.tolist(),
        "training_silhouette_score": float(silhouette_score(train_pc, train_labels)),
        "test_data_loaded": False,
        "model_arrays_sha256": hashlib.sha256(MODEL_DATA.read_bytes()).hexdigest(),
        "salary_data_sha256": hashlib.sha256(SALARY_DATA.read_bytes()).hexdigest(),
        "interpretation": "Exploratory descriptive groups jointly reflect performance and salary; not causal or a dollar valuation.",
    }
    (RESULTS / "analysis_audit.json").write_text(
        json.dumps(metadata, indent=2, allow_nan=False) + "\n", encoding="utf-8"
    )
    print(f"Training player-seasons used: {len(train_fit)} / {len(train)}")
    print(f"Development player-seasons assigned: {len(dev_fit)} / {len(dev)}")
    print(f"Training salary-flagged rows included: {metadata['training_salary_flagged_rows_used']}")
    print(f"Development salary-flagged rows included: {metadata['development_salary_flagged_rows_used']}")
    print(f"Training silhouette: {metadata['training_silhouette_score']:.4f}")
    print(pd.DataFrame(summaries).to_string(index=False))
    print("Test split not loaded. Results: " + str(RESULTS))

doSalaryAwareClusters = False
if doSalaryAwareClusters: runSalaryAwareClusters()
print('Salary-aware PCA clusters enabled: ' + str(doSalaryAwareClusters))

## 5. Role-crossing comps

Set the switch to `True` to run this analysis. Prerequisite files and output location are defined in the implementation below.


In [ ]:
"""Compare bench-player skill profiles with historical starter profiles.

Similarity is based on age, position, per-36 box-score rates, efficiency, usage,
and rate-based impact metrics. Playing time and salary are deliberately excluded
from the distance; playing time defines bench/starter roles, while salary and a
training-only salary-value residual are reported as context.
"""

from pathlib import Path
import hashlib
import json

import numpy as np
import pandas as pd
from sklearn.decomposition import PCA
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler

ROOT = rootPath
PROCESSED = ROOT / "data" / "processed"
SALARY_VALUE = ROOT / "results" / "development" / "salary_value"
RESULTS = ROOT / "results" / "development" / "role_crossing_comps"
SALARY_DATA = PROCESSED / "salary_cap_normalized_pairs.csv"
N_COMPS = 10
PCA_VARIANCE = 0.90
BENCH_MAX_MPG = 24.0
STARTER_MIN_MPG = 28.0
MIN_SEASON_MINUTES = 300.0
TARGETS = ["pie", "ws", "vorp"]
COUNTING_PER_GAME = [
    "pts_per_game", "trb_per_game", "ast_per_game",
    "stl_per_game", "blk_per_game", "tov_per_game",
]
RATE_FEATURES = [
    "ts_percent", "x3p_percent", "x3p_ar", "f_tr", "usg_percent",
    "per", "ws_48", "bpm", "pie",
]


def prepare_features(frame, position_columns):
    """Create skill/style inputs without minutes, per-game volume, or salary."""
    out = pd.DataFrame(index=frame.index)
    out["age"] = pd.to_numeric(frame["age"], errors="coerce")
    minutes_per_game = pd.to_numeric(frame["mp_per_game"], errors="coerce")
    total_minutes = pd.to_numeric(frame["mp"], errors="coerce")
    for column in COUNTING_PER_GAME:
        per_game = pd.to_numeric(frame[column], errors="coerce")
        per36 = per_game.div(minutes_per_game.where(minutes_per_game.gt(0))).mul(36.0)
        out[column.replace("_per_game", "_per36")] = per36
    for column in RATE_FEATURES:
        out[column] = pd.to_numeric(frame[column], errors="coerce")
    positions = frame["pos"].fillna("UNK").astype(str).str.upper()
    for position in position_columns:
        out["pos_" + position] = positions.str.contains(position, regex=False).astype(float)
    return out


def attach_salary_context(frame, salary):
    fields = [
        "player_id", "target_season", "salary_cap_share",
        "salary_available_check", "salary_needs_source_review",
    ]
    salary_fields = salary[fields].rename(
        columns={
            "salary_available_check": "normalized_salary_available",
            "salary_needs_source_review": "normalized_salary_review_flag",
        }
    )
    return frame.merge(
        salary_fields,
        on=["player_id", "target_season"],
        how="left",
        validate="one_to_one",
    )


def runRoleCrossingComps():
    RESULTS.mkdir(parents=True, exist_ok=True)
    train = pd.read_csv(PROCESSED / "train_rows.csv")
    dev = pd.read_csv(PROCESSED / "dev_rows.csv")
    salary = pd.read_csv(SALARY_DATA)
    value_train = pd.read_csv(SALARY_VALUE / "training_salary_benchmark_rows.csv")
    salary_keys = ["player_id", "target_season"]
    train = attach_salary_context(train, salary)
    dev = attach_salary_context(dev, salary)
    train = train.merge(
        value_train[salary_keys + ["observed_value_residual"]].rename(
            columns={"observed_value_residual": "salary_value_residual"}
        ),
        on=salary_keys,
        how="left",
        validate="one_to_one",
    )

    positions = sorted(
        set(train["pos"].dropna().astype(str).str.upper())
        | set(dev["pos"].dropna().astype(str).str.upper())
    )
    train_features = prepare_features(train, positions)
    dev_features = prepare_features(dev, positions)
    train_reliability = (
        pd.to_numeric(train["mp"], errors="coerce").clip(lower=0)
        / (pd.to_numeric(train["mp"], errors="coerce").clip(lower=0) + 600.0)
    ).fillna(0).to_numpy()
    dev_reliability = (
        pd.to_numeric(dev["mp"], errors="coerce").clip(lower=0)
        / (pd.to_numeric(dev["mp"], errors="coerce").clip(lower=0) + 600.0)
    ).fillna(0).to_numpy()
    for column in [name.replace("_per_game", "_per36") for name in COUNTING_PER_GAME]:
        league_mean = float(train_features[column].mean())
        train_features[column] = (
            train_features[column] * train_reliability
            + league_mean * (1 - train_reliability)
        )
        dev_features[column] = (
            dev_features[column] * dev_reliability
            + league_mean * (1 - dev_reliability)
        )
    imputer = SimpleImputer(strategy="median")
    train_imputed = imputer.fit_transform(train_features)
    dev_imputed = imputer.transform(dev_features)
    scaler = StandardScaler()
    train_scaled = scaler.fit_transform(train_imputed)
    dev_scaled = scaler.transform(dev_imputed)

    train_usable = (
        train["mp"].ge(MIN_SEASON_MINUTES).to_numpy()
        & np.isfinite(train_scaled).all(axis=1)
        & train[["next_season_" + target for target in TARGETS]]
        .apply(pd.to_numeric, errors="coerce").notna().all(axis=1).to_numpy()
    )
    dev_usable = (
        dev["mp"].ge(MIN_SEASON_MINUTES).to_numpy()
        & np.isfinite(dev_scaled).all(axis=1)
        & dev[["next_season_" + target for target in TARGETS]]
        .apply(pd.to_numeric, errors="coerce").notna().all(axis=1).to_numpy()
    )
    train = train.loc[train_usable].reset_index(drop=True)
    dev = dev.loc[dev_usable].reset_index(drop=True)
    train_scaled = train_scaled[train_usable]
    dev_scaled = dev_scaled[dev_usable]

    pca = PCA(n_components=PCA_VARIANCE, svd_solver="full")
    train_pc = pca.fit_transform(train_scaled)
    dev_pc = pca.transform(dev_scaled)
    coordinate_columns = [f"pc{i + 1}" for i in range(pca.n_components_)]
    train_coordinates = train[
        ["player", "player_id", "season", "target_season", "mp_per_game"]
    ].copy()
    train_coordinates["split"] = "training"
    train_coordinates["role"] = np.where(
        train_coordinates["mp_per_game"] >= STARTER_MIN_MPG, "starter",
        np.where(train_coordinates["mp_per_game"] < BENCH_MAX_MPG, "bench", "middle")
    )
    train_coordinates[coordinate_columns] = train_pc
    dev_coordinates = dev[
        ["player", "player_id", "season", "target_season", "mp_per_game"]
    ].copy()
    dev_coordinates["split"] = "development"
    dev_coordinates["role"] = np.where(
        dev_coordinates["mp_per_game"] >= STARTER_MIN_MPG, "starter",
        np.where(dev_coordinates["mp_per_game"] < BENCH_MAX_MPG, "bench", "middle")
    )
    dev_coordinates[coordinate_columns] = dev_pc
    pd.concat([train_coordinates, dev_coordinates], ignore_index=True).to_csv(
        RESULTS / "skill_pca_coordinates.csv", index=False
    )
    target_cols = ["next_season_" + target for target in TARGETS]
    deltas = train[target_cols].to_numpy(float) - train[TARGETS].to_numpy(float)

    valid_salary = (
        train["normalized_salary_available"].fillna(False).astype(bool)
        & ~train["normalized_salary_review_flag"].fillna(True).astype(bool)
        & train["salary_cap_share"].notna()
    )
    high_salary_threshold = float(train.loc[valid_salary, "salary_cap_share"].quantile(0.75))

    neighbors = []
    summaries = []
    for query_index, query in dev.iterrows():
        if not (query["mp_per_game"] < BENCH_MAX_MPG):
            continue
        available = (
            train["mp_per_game"].ge(STARTER_MIN_MPG)
            & train["target_season"].le(query["season"])
            & train["player_id"].ne(query["player_id"])
        ).to_numpy()
        candidate_indices = np.flatnonzero(available)
        if not len(candidate_indices):
            continue
        distances = np.linalg.norm(
            train_pc[candidate_indices] - dev_pc[query_index], axis=1
        )
        ranked = candidate_indices[np.argsort(distances, kind="stable")]
        chosen = []
        seen = set()
        for index in ranked:
            player_id = train.iloc[index]["player_id"]
            if player_id in seen:
                continue
            chosen.append(index)
            seen.add(player_id)
            if len(chosen) == N_COMPS:
                break
        if not chosen:
            continue

        query_summary = {
            "bench_player": query["player"],
            "bench_player_id": query["player_id"],
            "prior_season": int(query["season"]),
            "target_season": int(query["target_season"]),
            "prior_minutes_per_game": float(query["mp_per_game"]),
            "prior_total_minutes": float(query["mp"]),
            "prior_salary_cap_share_pct": (
                float(query["salary_cap_share"] * 100)
                if pd.notna(query["salary_cap_share"]) else np.nan
            ),
            "starter_comps_found": len(chosen),
            "high_salary_threshold_training_p75_pct": high_salary_threshold * 100,
        }
        for j, target in enumerate(TARGETS):
            query_summary["bench_prior_" + target] = float(query[target])
            query_summary["bench_actual_next_" + target] = float(query[target_cols[j]])
            query_summary["median_starter_comp_delta_" + target] = float(
                np.median(deltas[chosen, j])
            )
        summaries.append(query_summary)

        for rank, index in enumerate(chosen, start=1):
            comp = train.iloc[index]
            share = comp["salary_cap_share"]
            residual = comp["salary_value_residual"]
            high_paid_under_benchmark = bool(
                pd.notna(share)
                and pd.notna(residual)
                and share >= high_salary_threshold
                and residual < 0
            )
            item = {
                "bench_player": query["player"],
                "bench_player_id": query["player_id"],
                "bench_prior_season": int(query["season"]),
                "bench_target_season": int(query["target_season"]),
                "bench_prior_minutes_per_game": float(query["mp_per_game"]),
                "comp_rank": rank,
                "skill_distance_in_pca_space": float(
                    np.linalg.norm(train_pc[index] - dev_pc[query_index])
                ),
                "starter_comp": comp["player"],
                "starter_comp_player_id": comp["player_id"],
                "comp_prior_season": int(comp["season"]),
                "comp_outcome_season": int(comp["target_season"]),
                "comp_prior_minutes_per_game": float(comp["mp_per_game"]),
                "comp_salary_cap_share_pct": (
                    float(share * 100) if pd.notna(share) else np.nan
                ),
                "comp_salary_review_flag": bool(
                    comp["normalized_salary_review_flag"]
                ) if pd.notna(comp["normalized_salary_review_flag"]) else True,
                "comp_salary_performance_residual": (
                    float(residual) if pd.notna(residual) else np.nan
                ),
                "high_paid_below_benchmark_proxy": high_paid_under_benchmark,
            }
            for j, target in enumerate(TARGETS):
                item["comp_prior_" + target] = float(comp[target])
                item["comp_next_" + target] = float(comp[target_cols[j]])
                item["comp_delta_" + target] = float(deltas[index, j])
            neighbors.append(item)

    neighbor_frame = pd.DataFrame(neighbors)
    summary_frame = pd.DataFrame(summaries)
    neighbor_frame.to_csv(RESULTS / "development_bench_to_starter_comps.csv", index=False)
    summary_frame.to_csv(RESULTS / "development_bench_comp_summary.csv", index=False)

    metric_rows = []
    for target in TARGETS:
        actual = summary_frame["bench_actual_next_" + target].to_numpy(float)
        prior = summary_frame["bench_prior_" + target].to_numpy(float)
        analog = prior + summary_frame["median_starter_comp_delta_" + target].to_numpy(float)
        metric_rows.append(
            {
                "target": target,
                "bench_development_rows": int(len(summary_frame)),
                "starter_comp_median_delta_rmse": float(np.sqrt(np.mean((analog - actual) ** 2))),
                "starter_comp_median_delta_mae": float(np.mean(np.abs(analog - actual))),
                "persistence_rmse": float(np.sqrt(np.mean((prior - actual) ** 2))),
                "persistence_mae": float(np.mean(np.abs(prior - actual))),
            }
        )
    pd.DataFrame(metric_rows).to_csv(RESULTS / "development_bench_comp_metrics.csv", index=False)

    high_value_comps = neighbor_frame["high_paid_below_benchmark_proxy"].sum() if len(neighbor_frame) else 0
    audit = {
        "method": "PCA-reduced nearest skill-profile matching from bench queries to historical starter candidates",
        "similarity_features": list(train_features.columns),
        "excluded_from_similarity": [
            "minutes and games (role filter only)",
            "per-game counting totals (converted to shrunken per-36 rates)",
            "salary (reported as context only)",
        ],
        "minutes_shrinkage": "per-36 rates shrunk toward the training feature mean with reliability total_minutes / (total_minutes + 600)",
        "pca_variance_retained": float(pca.explained_variance_ratio_.sum()),
        "pca_components_retained": int(pca.n_components_),
        "variance_by_component": pca.explained_variance_ratio_.tolist(),
        "bench_definition": f"prior-season minutes per game < {BENCH_MAX_MPG}; at least {MIN_SEASON_MINUTES} total minutes",
        "starter_definition": f"prior-season minutes per game >= {STARTER_MIN_MPG}; at least {MIN_SEASON_MINUTES} total minutes",
        "analog_time_rule": "starter comp target season must be no later than the bench player's prior season",
        "distinct_starter_players": True,
        "starter_comps_per_query_max": N_COMPS,
        "salary_note": "Salary excluded from skill distance. High-paid/below-benchmark is a proxy using training 75th percentile salary-cap share and negative salary-value residual; it is not a claim of overpayment.",
        "high_salary_cap_share_threshold": high_salary_threshold,
        "training_rows_after_minute_filter": int(len(train)),
        "development_rows_after_minute_filter": int(len(dev)),
        "development_bench_queries_with_starter_comps": int(len(summary_frame)),
        "starter_comps_flagged_high_paid_below_benchmark": int(high_value_comps),
        "salary_normalized_file_sha256": hashlib.sha256(SALARY_DATA.read_bytes()).hexdigest(),
        "test_data_loaded": False,
        "interpretation": "Exploratory role-crossing skill comps; development results are not independent because development informed earlier project decisions.",
    }
    (RESULTS / "analysis_audit.json").write_text(
        json.dumps(audit, indent=2, allow_nan=False) + "\n", encoding="utf-8"
    )
    print(f"Skill features: {len(train_features.columns)}; PCA components: {pca.n_components_} "
          f"({audit['pca_variance_retained']:.1%} variance)")
    print(f"Development bench queries matched to starters: {len(summary_frame)}")
    print(f"High-salary, below-benchmark comp flags: {high_value_comps}")
    print(pd.read_csv(RESULTS / "development_bench_comp_metrics.csv").to_string(index=False))
    print("Results: " + str(RESULTS))

doRoleCrossingComps = False
if doRoleCrossingComps: runRoleCrossingComps()
print('Role-crossing comps enabled: ' + str(doRoleCrossingComps))

## 6. Role-crossing comp plots

Set the switch to `True` to run this analysis. Prerequisite files and output location are defined in the implementation below.


In [ ]:
"""Plot bench and starter skill profiles and one example set of comps."""

from pathlib import Path
import matplotlib.pyplot as plt
import pandas as pd

ROOT = rootPath
RESULTS = ROOT / "results" / "development" / "role_crossing_comps"


def plotRoleCrossingComps():
    coords = pd.read_csv(RESULTS / "skill_pca_coordinates.csv")
    comps = pd.read_csv(RESULTS / "development_bench_to_starter_comps.csv")
    queries = coords[
        (coords["split"] == "development")
        & (coords["role"] == "bench")
    ].copy()
    example = queries[queries["player"].eq("Isaiah Jackson")]
    if example.empty:
        example = queries.head(1)
    query = example.iloc[0]
    selected_comps = comps[
        (comps["bench_player_id"] == query["player_id"])
        & (comps["bench_prior_season"] == query["season"])
    ].sort_values("comp_rank").head(3)
    training_starters = coords[
        (coords["split"] == "training")
        & (coords["role"] == "starter")
    ]
    development_bench = coords[
        (coords["split"] == "development")
        & (coords["role"] == "bench")
    ]
    training_middle = coords[coords["split"].eq("training") & coords["role"].eq("middle")]

    fig, ax = plt.subplots(figsize=(10, 7), dpi=170)
    ax.scatter(
        training_starters["pc1"], training_starters["pc2"],
        s=13, alpha=0.13, color="#888888", label="Training starters",
    )
    ax.scatter(
        training_middle["pc1"], training_middle["pc2"],
        s=10, alpha=0.10, color="#C5A96B", label="Training middle role",
    )
    ax.scatter(
        development_bench["pc1"], development_bench["pc2"],
        s=18, alpha=0.32, color="#3977A8", label="Development bench players",
    )
    ax.scatter(
        query["pc1"], query["pc2"], marker="*", s=240,
        color="#111111", edgecolor="white", linewidth=0.8, zorder=5,
        label=f"{query['player']} (bench query)",
    )

    training_coords = coords[coords["split"].eq("training")]
    for comp in selected_comps.itertuples():
        match = training_coords[
            (training_coords["player_id"] == comp.starter_comp_player_id)
            & (training_coords["target_season"] == comp.comp_outcome_season)
        ]
        if match.empty:
            continue
        point = match.iloc[0]
        ax.plot(
            [query["pc1"], point["pc1"]],
            [query["pc2"], point["pc2"]],
            color="#C45B55", alpha=0.65, linewidth=1.2, zorder=3,
        )
        ax.scatter(
            point["pc1"], point["pc2"], marker="D", s=65,
            color="#C45B55", edgecolor="white", linewidth=0.7, zorder=4,
            label=f"Top starter comp: {comp.starter_comp}",
        )
        ax.annotate(
            comp.starter_comp,
            (point["pc1"], point["pc2"]),
            xytext=(5, 4), textcoords="offset points", fontsize=8,
        )

    ax.annotate(
        str(query["player"]),
        (query["pc1"], query["pc2"]),
        xytext=(7, 8), textcoords="offset points", fontsize=9, weight="bold",
    )
    ax.axhline(0, color="#777777", linewidth=0.7, alpha=0.35)
    ax.axvline(0, color="#777777", linewidth=0.7, alpha=0.35)
    ax.set_xlabel("PC1")
    ax.set_ylabel("PC2")
    ax.set_title("Bench-to-starter skill-profile comparisons")
    ax.text(
        0.02, 0.98,
        f"{query['player']}, prior season {int(query['season'])}\n"
        "Projection only: nearest comps use all retained PCs, not just PC1/PC2",
        transform=ax.transAxes, va="top", ha="left", fontsize=9,
        bbox={"facecolor": "white", "alpha": 0.85, "edgecolor": "none", "pad": 5},
    )
    handles, labels = ax.get_legend_handles_labels()
    unique = dict(zip(labels, handles))
    ax.legend(
        unique.values(), unique.keys(), loc="best", fontsize=8,
        frameon=True, ncol=2,
    )
    ax.grid(alpha=0.13)
    fig.tight_layout()
    fig.savefig(RESULTS / "bench_to_starter_skill_pca.png", bbox_inches="tight")
    fig.savefig(RESULTS / "bench_to_starter_skill_pca.pdf", bbox_inches="tight")
    print(RESULTS / "bench_to_starter_skill_pca.png")

runtRoleCrossingComps = False
if runtRoleCrossingComps: plotRoleCrossingComps()
print('Role-crossing comp plots enabled: ' + str(runtRoleCrossingComps))

## 7. Salary-aware PCA plots

Set the switch to `True` to run this analysis. Prerequisite files and output location are defined in the implementation below.


In [ ]:
"""Plot salary-aware PCA groups in their first two principal components."""

from pathlib import Path
import json

import matplotlib.pyplot as plt
import pandas as pd

ROOT = rootPath
RESULTS = ROOT / "results" / "development" / "salary_aware_pca"
COLORS = {
    "Profile 1": "#3977A8",
    "Profile 2": "#E0A43A",
    "Profile 3": "#4A8B69",
    "Profile 4": "#C45B55",
}


def plotSalaryAwarePca():
    rows = pd.read_csv(RESULTS / "salary_aware_player_groups.csv")
    audit = json.loads((RESULTS / "analysis_audit.json").read_text(encoding="utf-8"))
    fig, ax = plt.subplots(figsize=(10, 7), dpi=170)

    for profile, color in COLORS.items():
        for split, marker, size, alpha in (
            ("training", "o", 15, 0.20),
            ("development", "^", 30, 0.60),
        ):
            selected = rows[(rows["profile"] == profile) & (rows["split"] == split)]
            ax.scatter(
                selected["pc1"],
                selected["pc2"],
                c=color,
                marker=marker,
                s=size,
                alpha=alpha,
                linewidths=0,
                label=f"{profile} — {split.title()}",
            )

    ratios = audit["explained_variance_ratio"]
    ax.axhline(0, color="#777777", linewidth=0.7, alpha=0.4)
    ax.axvline(0, color="#777777", linewidth=0.7, alpha=0.4)
    ax.set_xlabel(f"PC1 ({ratios[0] * 100:.1f}% variance)")
    ax.set_ylabel(f"PC2 ({ratios[1] * 100:.1f}% variance)")
    ax.set_title("Salary-aware PCA player-season groups", fontsize=16, pad=14)
    ax.text(
        0.02,
        0.98,
        "PCA fit on training rows; development rows projected onto same axes\n"
        "Inputs: next-season PIE, WS, VORP, log salary / salary cap",
        transform=ax.transAxes,
        va="top",
        ha="left",
        fontsize=9,
        color="#444444",
        bbox={"facecolor": "white", "alpha": 0.85, "edgecolor": "none", "pad": 5},
    )
    handles, labels = ax.get_legend_handles_labels()
    unique = dict(zip(labels, handles))
    ax.legend(unique.values(), unique.keys(), loc="best", frameon=True, fontsize=8, ncol=2)
    ax.grid(alpha=0.15)
    fig.tight_layout()
    fig.savefig(RESULTS / "salary_aware_pca_profiles.png", bbox_inches="tight")
    fig.savefig(RESULTS / "salary_aware_pca_profiles.pdf", bbox_inches="tight")
    print(RESULTS / "salary_aware_pca_profiles.png")

runtSalaryAwarePca = False
if runtSalaryAwarePca: plotSalaryAwarePca()
print('Salary-aware PCA plots enabled: ' + str(runtSalaryAwarePca))

## 9. View generated exploratory figures inline


In [ ]:
optionalFigureRoots = [
    rootPath / 'results' / 'development' / 'player_comps',
    rootPath / 'results' / 'development' / 'role_crossing_comps',
    rootPath / 'results' / 'development' / 'salary_aware_pca',
    rootPath / 'results' / 'development' / 'salary_value',
]
optionalFigures = sorted({image for folder in optionalFigureRoots if folder.exists() for image in folder.rglob('*.png')})
if optionalFigures:
    for imagePath in optionalFigures:
        print(imagePath.relative_to(rootPath))
        if Image is not None: display(Image(filename=str(imagePath)))
else:
    print('No optional-analysis figures found yet; run a relevant analysis above.')

## Interpretation and scope

- These analyses are for exploration and visualization, not model selection against a held-out test set.
- The salary-value shortlist analysis and historical comps use older single-target/PCA artifacts and may not align with the current 16-output profile model.
- Plots are shown from saved artifacts so the notebook remains useful when long analyses are not rerun.
